In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q langchain langchain-chroma langchain-huggingface chromadb langchain-google-genai

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   

In [2]:
import json
import os

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

JSON_FILE_PATH = os.path.join(DRIVE_FOLDER, "knowledge_base.json")

data = [
  {
    "subject": "Khoa học tự nhiên",
    "grade": "Lớp 6",
    "topic": "Tế bào",
    "lesson": "Bài 18: Tế bào – Đơn vị cơ bản của sự sống",
    "concept": "Tế bào",
    "content": "Mọi cơ thể sống đều được cấu tạo từ tế bào. Tế bào thực hiện các chức năng sống cơ bản như trao đổi chất, sinh trưởng, phát triển và sinh sản.",
    "simple_explanation": "Tế bào là đơn vị cấu tạo cơ bản của cơ thể sống, giống như những viên gạch nhỏ tạo nên một ngôi nhà.",
    "example": "Cây cối, con người, cá, chim đều được cấu tạo từ hàng triệu tế bào.",
    "question_answer": "H: Tế bào là gì?\nĐ: Tế bào là đơn vị cơ bản cấu tạo nên mọi cơ thể sinh vật.",
    "difficulty": "Dễ",
    "source": "SGK Khoa học tự nhiên 6 - Trang 64"
  },
  {
    "subject": "Công nghệ",
    "grade": "Lớp 6",
    "topic": "Nhà ở",
    "lesson": "Bài 3: Ngôi nhà thông minh",
    "concept": "Ngôi nhà thông minh",
    "content": "Ngôi nhà thông minh là ngôi nhà được trang bị hệ thống điều khiển tự động hoặc bán tự động cho các thiết bị trong nhà, giúp cuộc sống trở nên tiện nghi, an toàn và tiết kiệm năng lượng.",
    "simple_explanation": "Là ngôi nhà có các thiết bị điện tự biết hoạt động khi cần thiết mà người dùng không bắt buộc phải bấm nút thủ công.",
    "example": "Đèn tự bật sáng khi em bước vào phòng và tự tắt khi em đi ra ngoài.",
    "question_answer": "H: Ngôi nhà thông minh giúp ích gì cho con người?\nĐ: Giúp cuộc sống tiện nghi, an toàn và tiết kiệm điện năng hơn.",
    "difficulty": "Trung bình",
    "source": "SGK Công nghệ 6 - Trang 18"
  }
]

with open(JSON_FILE_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Đã tạo thành công file dữ liệu tại: {JSON_FILE_PATH}")

✅ Đã tạo thành công file dữ liệu tại: /content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH/knowledge_base.json


In [3]:
import os
import json
import torch
from typing import List, Tuple
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
JSON_FILE_PATH = os.path.join(DRIVE_FOLDER, "knowledge_base.json")
DB_DIR = os.path.join(DRIVE_FOLDER, "chroma_db")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Đang sử dụng thiết bị tính toán: {DEVICE.upper()}")

def load_structured_knowledge(json_path: str) -> Tuple[List[Document], List[str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents, ids = [], []
    for idx, item in enumerate(data):
        page_content = (
            f"Môn học: {item.get('subject', '')}\n"
            f"Chủ đề: {item.get('topic', '')}\n"
            f"Khái niệm: {item.get('concept', '')}\n"
            f"Giải thích đơn giản: {item.get('simple_explanation', '')}\n"
            f"Nội dung chuẩn SGK: {item.get('content', '')}\n"
            f"Ví dụ thực tế: {item.get('example', '')}\n"
            f"Hỏi đáp mẫu:\n{item.get('question_answer', '')}"
        )

        metadata = {
            "subject": item.get("subject", ""),
            "grade": str(item.get("grade", "Lớp 6")),
            "topic": item.get("topic", ""),
            "lesson": item.get("lesson", ""),
            "concept": item.get("concept", ""),
            "difficulty": item.get("difficulty", "Dễ"),
            "source": item.get("source", ""),
            "simple_explanation": item.get("simple_explanation", ""),
            "example": item.get("example", "")
        }

        doc_id = f"kb_{item.get('subject', 'general')}_{idx}"
        documents.append(Document(page_content=page_content, metadata=metadata))
        ids.append(doc_id)

    return documents, ids

docs, doc_ids = load_structured_knowledge(JSON_FILE_PATH)

embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': DEVICE}
)

vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    ids=doc_ids,
    persist_directory=DB_DIR
)

print(f"✅ Đã lưu ChromaDB thành công vào Google Drive tại: {DB_DIR}")

🚀 Đang sử dụng thiết bị tính toán: CPU


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (6 outputs left out)

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

✅ Đã lưu ChromaDB thành công vào Google Drive tại: /content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH/chroma_db


In [4]:
!pip install -q -U google-genai

In [5]:
import json
import os
import re
import time
import uuid
from typing import List, Tuple
from google import genai

API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJt6RZUsnS7RD1hnST_ZxIA"

client = genai.Client(api_key=API_KEY)

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
os.makedirs(DRIVE_FOLDER, exist_ok=True)
OUTPUT_JSON_PATH = os.path.join(DRIVE_FOLDER, "knowledge_base.json")

def generate_with_smart_fallback(prompt: str) -> str:
    """Tự động dò model khả dụng, lách lỗi limit: 0 của Free Tier"""

    candidate_models = [
        "gemini-2.5-flash-lite",
        "gemini-2.0-flash-lite",
        "gemini-3.5-flash-lite",
        "gemini-2.5-flash"
    ]

    last_error = None
    for model_name in candidate_models:
        print(f"     Đang kết nối model: {model_name}...")

        for attempt in range(2):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt
                )
                if response and response.text:
                    return response.text

            except Exception as e:
                error_msg = str(e)
                last_error = e


                if "limit: 0" in error_msg:
                    print(f"     Model {model_name} bị khóa limit: 0. Đang đổi model khác...")
                    break

                elif "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                    print(f"     Quá tải tốc độ. Chờ 10 giây...")
                    time.sleep(10)
                else:
                    break

    raise RuntimeError(f"Tất cả các model đều thất bại. Lỗi cuối cùng: {last_error}")

def audit_knowledge_item(item: dict) -> Tuple[bool, List[str]]:
    errors = []
    required_fields = [
        "id", "subject", "grade", "topic", "lesson", "concept",
        "content", "simple_explanation", "example",
        "question_answer", "keywords", "difficulty", "source"
    ]

    for field in required_fields:
        if field not in item:
            errors.append(f"Thiếu trường '{field}'")
        elif not item[field]:
            errors.append(f"Trường '{field}' bị trống")

    if item.get("subject") not in ["Khoa học tự nhiên", "Công nghệ"]:
        errors.append("Trường 'subject' sai định dạng.")

    if not isinstance(item.get("keywords"), list):
        errors.append("Trường 'keywords' phải là mảng.")

    qa = item.get("question_answer", {})
    if not isinstance(qa, dict) or "question" not in qa or "answer" not in qa:
        errors.append("Trường 'question_answer' phải là object chứa question và answer.")

    return len(errors) == 0, errors

def process_raw_text_to_knowledge_item(
    raw_text: str, subject: str, topic: str, lesson: str, source_page: str
) -> dict:
    prompt = f"""
    Bạn là chuyên gia biên soạn chương trình giáo dục Lớp 6.
    Trích xuất đoạn văn bản thô thuộc môn '{subject}', '{lesson}' thành JSON thuần túy (Không dùng Markdown block):

    ĐOẠN VĂN BẢN THÔ:
    "{raw_text}"

    CẤU TRÚC JSON YÊU CẦU:
    {{
        "subject": "{subject}",
        "grade": "Lớp 6",
        "topic": "{topic}",
        "lesson": "{lesson}",
        "concept": "<Khái niệm chính>",
        "content": "<Nội dung chuẩn xác>",
        "simple_explanation": "<Giải thích đơn giản>",
        "example": "<Ví dụ thực tế>",
        "question_answer": {{
            "question": "<Câu hỏi ôn tập>",
            "answer": "<Trả lời>"
        }},
        "keywords": ["<từ khóa 1>", "<từ khóa 2>"],
        "difficulty": "Dễ",
        "source": "{source_page}"
    }}
    """

    try:
        response_text = generate_with_smart_fallback(prompt)
        match = re.search(r"\{.*\}", response_text, re.DOTALL)
        if match:
            json_data = json.loads(match.group(0))
            json_data["id"] = f"doc_{uuid.uuid4().hex[:12]}"
            return json_data
        else:
            return None
    except Exception as e:
        print(f"   Lỗi AI: {e}")
        return None

if __name__ == "__main__":
    print("=== BẮT ĐẦU PIPELINE BƯỚC 2: TIỀN XỬ LÝ (RAG READY) ===")

    raw_inputs = [
        {
            "raw_text": "Thuỷ ngân trong nhiệt kế y tế là chất độc dễ bay hơi. Khi sử dụng cần cẩn thận tránh làm vỡ. Nếu lỡ làm vỡ nhiệt kế, không được sờ tay vào thuỷ ngân mà phải báo ngay cho giáo viên hoặc người lớn.",
            "subject": "Khoa học tự nhiên",
            "topic": "Đo nhiệt độ",
            "lesson": "Bài 8: Đo nhiệt độ",
            "source_page": "SGK Khoa học tự nhiên 6 - Trang 26",
        },
        {
            "raw_text": "Ngôi nhà thông minh là ngôi nhà được trang bị hệ thống điều khiển tự động hoặc bán tự động cho các thiết bị trong nhà. Giúp tiết kiệm năng lượng, đảm bảo an toàn và mang lại sự tiện nghi cho người sử dụng.",
            "subject": "Công nghệ",
            "topic": "Nhà ở",
            "lesson": "Bài 3: Ngôi nhà thông minh",
            "source_page": "SGK Công nghệ 6 - Trang 18",
        },
    ]

    processed_database = []

    for idx, inp in enumerate(raw_inputs):
        print(f"\n⚙️ Đang xử lý mục {idx + 1}/{len(raw_inputs)}: {inp['lesson']}...")

        item = process_raw_text_to_knowledge_item(
            raw_text=inp["raw_text"],
            subject=inp["subject"],
            topic=inp["topic"],
            lesson=inp["lesson"],
            source_page=inp["source_page"],
        )

        if item:
            is_valid, errors = audit_knowledge_item(item)
            if is_valid:
                print(f"   Thành công! Sinh ID: {item['id']}")
                processed_database.append(item)
            else:
                print(f"   Lỗi định dạng JSON: {errors}")

        time.sleep(3)

    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(processed_database, f, ensure_ascii=False, indent=2)

    print(f"\n HOÀN THÀNH BƯỚC 2! Đã lưu {len(processed_database)} mục tri thức vào file.")
    print(f" File JSON: {OUTPUT_JSON_PATH}")

=== BẮT ĐẦU PIPELINE BƯỚC 2: TIỀN XỬ LÝ (RAG READY) ===

⚙️ Đang xử lý mục 1/2: Bài 8: Đo nhiệt độ...
     Đang kết nối model: gemini-2.5-flash-lite...
     Đang kết nối model: gemini-2.0-flash-lite...
     Đang kết nối model: gemini-3.5-flash-lite...
   Thành công! Sinh ID: doc_dcea484ca6e7

⚙️ Đang xử lý mục 2/2: Bài 3: Ngôi nhà thông minh...
     Đang kết nối model: gemini-2.5-flash-lite...
     Đang kết nối model: gemini-2.0-flash-lite...
     Đang kết nối model: gemini-3.5-flash-lite...
   Thành công! Sinh ID: doc_a0cd5f7cad56

 HOÀN THÀNH BƯỚC 2! Đã lưu 2 mục tri thức vào file.
 File JSON: /content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH/knowledge_base.json


In [6]:
import json
import os
import re

# 1. CẤU HÌNH ĐƯỜNG DẪN FILE
DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
INPUT_JSON_PATH = os.path.join(DRIVE_FOLDER, "knowledge_base.json")
OUTPUT_CHUNK_PATH = os.path.join(DRIVE_FOLDER, "chunks_database.json")

SAFETY_KEYWORDS = ["cẩn thận", "độc", "nguy hiểm", "an toàn", "tránh", "không được", "chú ý"]

def process_semantic_chunking(data_list):
    chunks = []

    for item in data_list:
        doc_id = item.get("id", "unknown_id")

        keywords_str = ", ".join(item.get("keywords", [])) if isinstance(item.get("keywords"), list) else ""

        base_metadata = {
            "source_doc_id": doc_id,
            "subject": item.get("subject", ""),
            "grade": item.get("grade", ""),
            "topic": item.get("topic", ""),
            "lesson": item.get("lesson", ""),
            "difficulty": item.get("difficulty", ""),
            "keywords": keywords_str
        }

        if item.get("concept") or item.get("simple_explanation"):
            chunks.append({
                "chunk_id": f"{doc_id}_concept",
                "chunk_type": "Concept chunk",
                "content": f"Khái niệm: {item.get('concept', '')}. Giải thích dễ hiểu: {item.get('simple_explanation', '')}",
                "metadata": base_metadata
            })

        if item.get("example"):
            chunks.append({
                "chunk_id": f"{doc_id}_example",
                "chunk_type": "Example chunk",
                "content": f"Ví dụ thực tế: {item['example']}",
                "metadata": base_metadata
            })

        if item.get("question_answer"):
            qa = item["question_answer"]
            chunks.append({
                "chunk_id": f"{doc_id}_exercise",
                "chunk_type": "Exercise hint chunk",
                "content": f"Câu hỏi ôn tập: {qa.get('question', '')} - Đáp án/Gợi ý: {qa.get('answer', '')}",
                "metadata": base_metadata
            })

        if item.get("content"):
            chunks.append({
                "chunk_id": f"{doc_id}_summary",
                "chunk_type": "Summary chunk",
                "content": f"Nội dung trọng tâm: {item['content']}",
                "metadata": base_metadata
            })

        full_text = f"{item.get('content', '')}. {item.get('example', '')}. {item.get('simple_explanation', '')}"
        sentences = re.split(r'(?<=[.!?])\s+', full_text)

        safety_sentences = []
        for sentence in sentences:
            if any(kw in sentence.lower() for kw in SAFETY_KEYWORDS):
                safety_sentences.append(sentence.strip())

        if safety_sentences:
            safe_content = " ".join(safety_sentences)
            chunks.append({
                "chunk_id": f"{doc_id}_safety",
                "chunk_type": "Safety note chunk",
                "content": f"Lưu ý an toàn quan trọng: {safe_content}",
                "metadata": base_metadata
            })

    return chunks

if __name__ == "__main__":
    print("=== BẮT ĐẦU PIPELINE BƯỚC 3: SEMANTIC CHUNKING ===")

    if not os.path.exists(INPUT_JSON_PATH):
        print(f" Không tìm thấy file gốc tại: {INPUT_JSON_PATH}")
    else:
        with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        print(f" Đã tải {len(raw_data)} bài học. Đang tiến hành băm nhỏ (chunking)...")

        chunked_data = process_semantic_chunking(raw_data)

        with open(OUTPUT_CHUNK_PATH, "w", encoding="utf-8") as f:
            json.dump(chunked_data, f, ensure_ascii=False, indent=2)

        print(f" HOÀN THÀNH! Đã tạo ra {len(chunked_data)} chunks kiến thức độc lập.")
        print(" Thống kê các loại Chunk:")

        chunk_counts = {}
        for c in chunked_data:
            c_type = c["chunk_type"]
            chunk_counts[c_type] = chunk_counts.get(c_type, 0) + 1

        for c_type, count in chunk_counts.items():
            print(f"   - {c_type}: {count} chunks")

        print(f"\n File Chunk JSON đã lưu tại: {OUTPUT_CHUNK_PATH}")

=== BẮT ĐẦU PIPELINE BƯỚC 3: SEMANTIC CHUNKING ===
 Đã tải 2 bài học. Đang tiến hành băm nhỏ (chunking)...
 HOÀN THÀNH! Đã tạo ra 10 chunks kiến thức độc lập.
 Thống kê các loại Chunk:
   - Concept chunk: 2 chunks
   - Example chunk: 2 chunks
   - Exercise hint chunk: 2 chunks
   - Summary chunk: 2 chunks
   - Safety note chunk: 2 chunks

 File Chunk JSON đã lưu tại: /content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH/chunks_database.json


In [7]:
!pip install chromadb sentence-transformers

In [8]:
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
INPUT_CHUNK_PATH = os.path.join(DRIVE_FOLDER, "chunks_database.json")

CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")
os.makedirs(CHROMA_DB_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

print(" Đang tải mô hình Embedding đa ngữ (hỗ trợ Tiếng Việt)...")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

collection_name = "grade6_knowledge_base"

try:
    chroma_client.delete_collection(name=collection_name)
    print(f" Đã xóa Collection cũ: {collection_name}")
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)
print(f" Đã tạo Collection mới: {collection_name}")

def ingest_data_to_chroma():
    if not os.path.exists(INPUT_CHUNK_PATH):
        print(f" Không tìm thấy file dữ liệu: {INPUT_CHUNK_PATH}")
        return

    with open(INPUT_CHUNK_PATH, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f" Đang xử lý {len(chunks)} chunks để đưa vào Database...")

    ids = []
    documents = []
    metadatas = []

    for idx, chunk in enumerate(chunks):
        ids.append(chunk["chunk_id"])
        documents.append(chunk["content"])

        meta = chunk["metadata"].copy()
        meta["chunk_type"] = chunk["chunk_type"]
        metadatas.append(meta)

    print(" Đang nhúng (Embedding) văn bản thành vector. Vui lòng đợi...")
    embeddings = embedding_model.encode(documents).tolist()

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    print(f" THÀNH CÔNG! Đã nạp {len(ids)} vectors vào ChromaDB.")
    print(f" Cơ sở dữ liệu được lưu tại: {CHROMA_DB_PATH}")

if __name__ == "__main__":
    print("=== BẮT ĐẦU PIPELINE BƯỚC 4: EMBEDDING & VECTOR STORAGE ===")
    ingest_data_to_chroma()

    print("\n ĐANG KIỂM TRA TRUY XUẤT THỬ NGHIỆM (TEST):")
    test_query = "Làm sao để đo nhiệt độ an toàn?"
    print(f"Câu hỏi: '{test_query}'")

    query_embedding = embedding_model.encode([test_query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=2
    )

    for i in range(len(results['documents'][0])):
        print(f"\n   - Kết quả {i+1}:")
        print(f"     + Loại: {results['metadatas'][0][i]['chunk_type']}")
        print(f"     + Nội dung: {results['documents'][0][i]}")

 Đang tải mô hình Embedding đa ngữ (hỗ trợ Tiếng Việt)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Đã xóa Collection cũ: grade6_knowledge_base
 Đã tạo Collection mới: grade6_knowledge_base
=== BẮT ĐẦU PIPELINE BƯỚC 4: EMBEDDING & VECTOR STORAGE ===
 Đang xử lý 10 chunks để đưa vào Database...
 Đang nhúng (Embedding) văn bản thành vector. Vui lòng đợi...
 THÀNH CÔNG! Đã nạp 10 vectors vào ChromaDB.
 Cơ sở dữ liệu được lưu tại: /content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH/chroma_db

 ĐANG KIỂM TRA TRUY XUẤT THỬ NGHIỆM (TEST):
Câu hỏi: 'Làm sao để đo nhiệt độ an toàn?'

   - Kết quả 1:
     + Loại: Concept chunk
     + Nội dung: Khái niệm: An toàn khi sử dụng nhiệt kế chứa thủy ngân. Giải thích dễ hiểu: Thủy ngân rất độc và dễ bay hơi. Khi dùng nhiệt kế thủy ngân phải hết sức cẩn thận để không bị vỡ. Nếu chẳng may bị vỡ, tuyệt đối không chạm tay vào và phải nhờ người lớn hoặc thầy cô xử lý ngay.

   - Kết quả 2:
     + Loại: Safety note chunk
     + Nội dung: Lưu ý an toàn quan trọng: Thuỷ ngân trong nhiệt kế y tế là chất độc dễ bay hơi. Khi sử dụng cần cẩn thận tránh làm vỡ. Nếu lỡ làm 

In [9]:
import json
import re
from google import genai

API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJt6RZUsnS7RD1hnST_ZxIA"
client = genai.Client(api_key=API_KEY)

def analyze_student_query(query: str) -> dict:
    """
    Sử dụng AI để bóc tách ý định người dùng thành JSON định tuyến.
    """
    prompt = f"""
    Bạn là một module phân tích dữ liệu cho hệ thống Chatbot giáo dục lớp 6.
    Nhiệm vụ: Phân tích câu hỏi của học sinh: "{query}"

    Trả về MỘT CHUỖI JSON DUY NHẤT (không dùng markdown code block) theo đúng cấu trúc sau:
    {{
        "subject": "Công nghệ" hoặc "Khoa học tự nhiên" (Nếu câu hỏi quá chung chung không thể đoán được, ghi "Unknown"),
        "topic": "Tên chủ đề/bài học ngắn gọn" (Nếu không rõ, ghi "Unknown"),
        "question_type": "Khái niệm" / "So sánh" / "Vận dụng" / "Bài tập",
        "difficulty": "Dễ" / "Trung bình" / "Khó",
        "intent": "Giải thích" / "Tóm tắt" / "Gợi ý" / "Kiểm tra đáp án"
    }}
    """

    model_name = "gemini-3.5-flash-lite"

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=prompt
        )

        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return None

    except Exception as e:
        print(f" Lỗi AI Module Phân tích: {e}")
        return None

if __name__ == "__main__":
    print("=== BẮT ĐẦU PIPELINE BƯỚC 5: MODULE PHÂN TÍCH CÂU HỎI ===\n")

    test_queries = [
        "Thầy ơi, ngôi nhà thông minh khác gì nhà bình thường ạ?",
        "Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?",
        "Em muốn hỏi về năng lượng ạ."
    ]

    for q in test_queries:
        print(f" Học sinh: {q}")
        print(" Đang phân tích ý định...")

        analysis = analyze_student_query(q)

        if analysis:
            print(" JSON Trích xuất được:")
            print(json.dumps(analysis, ensure_ascii=False, indent=2))

            if analysis.get("subject") == "Unknown" or analysis.get("topic") == "Unknown":
                print(" HÀNH ĐỘNG:  Chatbot phản hồi: 'Em đang hỏi phần Công nghệ hay Khoa học tự nhiên vậy? Cho thầy/cô biết thêm chi tiết để hỗ trợ nhé!'")
            else:
                print(f" HÀNH ĐỘNG: Sẽ dùng metadata (subject='{analysis['subject']}') để chui vào ChromaDB tìm kiếm chính xác.")
        else:
            print(" Không thể phân tích câu hỏi.")

        print("-" * 50)

=== BẮT ĐẦU PIPELINE BƯỚC 5: MODULE PHÂN TÍCH CÂU HỎI ===

 Học sinh: Thầy ơi, ngôi nhà thông minh khác gì nhà bình thường ạ?
 Đang phân tích ý định...
 JSON Trích xuất được:
{
  "subject": "Công nghệ",
  "topic": "Ngôi nhà thông minh",
  "question_type": "So sánh",
  "difficulty": "Dễ",
  "intent": "Giải thích"
}
 HÀNH ĐỘNG: Sẽ dùng metadata (subject='Công nghệ') để chui vào ChromaDB tìm kiếm chính xác.
--------------------------------------------------
 Học sinh: Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?
 Đang phân tích ý định...
 JSON Trích xuất được:
{
  "subject": "Khoa học tự nhiên",
  "topic": "Sử dụng nhiệt kế an toàn",
  "question_type": "Vận dụng",
  "difficulty": "Trung bình",
  "intent": "Gợi ý"
}
 HÀNH ĐỘNG: Sẽ dùng metadata (subject='Khoa học tự nhiên') để chui vào ChromaDB tìm kiếm chính xác.
--------------------------------------------------
 Học sinh: Em muốn hỏi về năng lượng ạ.
 Đang phân tích ý định...
 JSON Trích xuất được:
{
  "subject": "Khoa học tự nhiên",


In [10]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")

print(" Đang kết nối với Cơ sở dữ liệu ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

try:
    collection = chroma_client.get_collection(name="grade6_knowledge_base")
    print(" Đã kết nối thành công với bộ nhớ: grade6_knowledge_base")
except Exception as e:
    print(f" Lỗi: Không tìm thấy Collection. Vui lòng kiểm tra lại Bước 4: {e}")

print(" Đang tải mô hình Embedding...")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def retrieve_knowledge(query: str, top_k: int = 3, distance_threshold: float = 20.0) -> list:
    """
    Truy xuất top_k chunks liên quan nhất.
    Có lọc theo Ngưỡng khoảng cách (distance_threshold) để loại bỏ kết quả 'rác'.
    """
    query_embedding = embedding_model.encode([query]).tolist()

    try:

        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )

        retrieved_chunks = []
        if results and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                dist = results['distances'][0][i]

                if dist <= distance_threshold:
                    chunk_data = {
                        "content": results['documents'][0][i],
                        "type": results['metadatas'][0][i].get('chunk_type', 'Unknown'),
                        "distance": dist
                    }
                    retrieved_chunks.append(chunk_data)

        return retrieved_chunks

    except Exception as e:
        print(f" Lỗi truy xuất: {e}")
        return []

if __name__ == "__main__":
    print("\n=== BẮT ĐẦU PIPELINE BƯỚC 6 (ĐÃ TỐI ƯU BỘ LỌC) ===")

    query_1 = "Nếu lỡ làm vỡ nhiệt kế thì phải làm sao?"
    print(f"\n👤 Test 1: '{query_1}'")
    results_1 = retrieve_knowledge(query_1)

    if results_1:
        print(f" Tìm thấy {len(results_1)} chunks sát nghĩa:")
        for res in results_1:
            print(f"   - [{res['type']}] (Độ lệch: {res['distance']:.2f}): {res['content'][:80]}...")
    else:
        print(" Không tìm thấy kiến thức phù hợp trong Database.")

    print("-" * 50)

    query_2 = "Vì sao cây cần ánh sáng?"
    print(f"\n👤 Test 2: '{query_2}'")
    results_2 = retrieve_knowledge(query_2)

    if results_2:
        print(f"Tìm thấy {len(results_2)} chunks sát nghĩa:")
        for res in results_2:
            print(f"   - [{res['type']}] (Độ lệch: {res['distance']:.2f}): {res['content'][:80]}...")
    else:
        print(" HỆ THỐNG PHÁT HIỆN: Chưa có bài học này trong cơ sở dữ liệu (Khoảng cách vector quá xa).")

 Đang kết nối với Cơ sở dữ liệu ChromaDB...
 Đã kết nối thành công với bộ nhớ: grade6_knowledge_base
 Đang tải mô hình Embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== BẮT ĐẦU PIPELINE BƯỚC 6 (ĐÃ TỐI ƯU BỘ LỌC) ===

👤 Test 1: 'Nếu lỡ làm vỡ nhiệt kế thì phải làm sao?'
 Tìm thấy 3 chunks sát nghĩa:
   - [Summary chunk] (Độ lệch: 7.91): Nội dung trọng tâm: Thuỷ ngân trong nhiệt kế y tế là chất độc dễ bay hơi. Khi sử...
   - [Safety note chunk] (Độ lệch: 8.53): Lưu ý an toàn quan trọng: Thuỷ ngân trong nhiệt kế y tế là chất độc dễ bay hơi. ...
   - [Exercise hint chunk] (Độ lệch: 8.57): Câu hỏi ôn tập: Cần làm gì nếu không may làm vỡ nhiệt kế thủy ngân? - Đáp án/Gợi...
--------------------------------------------------

👤 Test 2: 'Vì sao cây cần ánh sáng?'
 HỆ THỐNG PHÁT HIỆN: Chưa có bài học này trong cơ sở dữ liệu (Khoảng cách vector quá xa).


In [11]:
import os
import json
import re
import chromadb
from google import genai
from sentence_transformers import SentenceTransformer

API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJt6RZUsnS7RD1hnST_ZxIA"
client = genai.Client(api_key=API_KEY)

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_collection(name="grade6_knowledge_base")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def analyze_student_query(query: str) -> dict:
    prompt = f"""
    Bạn là module phân tích dữ liệu cho Chatbot giáo dục lớp 6.
    Phân tích câu hỏi: "{query}"
    Trả về MỘT CHUỖI JSON DUY NHẤT (không dùng markdown code block):
    {{
        "subject": "Công nghệ" hoặc "Khoa học tự nhiên" (Nếu không rõ ghi "Unknown"),
        "topic": "Tên chủ đề ngắn gọn" (Nếu không rõ ghi "Unknown"),
        "question_type": "Khái niệm" / "So sánh" / "Vận dụng" / "Bài tập",
        "intent": "Giải thích" / "Tóm tắt" / "Gợi ý" / "Kiểm tra đáp án"
    }}
    """
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )
        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        return json.loads(match.group(0)) if match else None
    except Exception:
        return None

def retrieve_knowledge(query: str, top_k: int = 3, distance_threshold: float = 20.0) -> list:
    query_embedding = embedding_model.encode([query]).tolist()
    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        retrieved_chunks = []
        if results and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                dist = results['distances'][0][i]
                if dist <= distance_threshold:
                    retrieved_chunks.append(results['documents'][0][i])
        return retrieved_chunks
    except Exception:
        return []

def rag_chatbot_pipeline(user_query: str):
    print(f"\n Học sinh hỏi: '{user_query}'")
    print(" [1/3] Đang phân tích câu hỏi...")

    analysis = analyze_student_query(user_query)

    if not analysis or analysis.get("subject") == "Unknown":
        print("\n Chatbot: 'Chào em! Câu hỏi của em hơi chung chung một chút. Em đang muốn hỏi kiến thức thuộc môn Công nghệ hay Khoa học tự nhiên lớp 6 vậy nhỉ? Cho thầy/cô biết thêm để hỗ trợ em tốt nhất nhé!'")
        return

    question_type = analysis.get("question_type", "Khái niệm")
    print(f"   -> Môn: {analysis['subject']} | Dạng: {question_type}")
    print(" [2/3] Đang tìm kiếm kiến thức trong kho sách...")

    context_chunks = retrieve_knowledge(user_query)

    if not context_chunks:
        print("\n Chatbot: 'Chào em! Kiến thức này hiện chưa có trong kho bài học lớp 6 của thầy/cô hoặc vượt quá chương trình học hiện tại. Em có thể hỏi lại thầy cô bộ môn trên lớp để được giải đáp chi tiết hơn nhé!'")
        return

    print(f"   -> Đã tìm thấy {len(context_chunks)} đoạn thông tin khớp.")
    print("⏳ [3/3] Đang sinh câu trả lời theo chuẩn Sư phạm...")

    context_text = "\n\n".join([f"- {c}" for c in context_chunks])

    pedagogical_prompt = f"""
    Bạn là một Chatbot học tập thân thiện, đóng vai giáo viên hướng dẫn học sinh lớp 6.
    Xưng hô: "thầy/cô" - "em".

    CÁC NGUYÊN TẮC SƯ PHẠM BẮT BUỘC:
    1. Chuẩn kiến thức Lớp 6: Chỉ sử dụng thông tin trong khối DỮ LIỆU SÁCH GIÁO KHOA dưới đây.
    2. Dễ hiểu & Ngắn gọn: Dùng câu ngắn, từ ngữ đơn giản. Nếu có thuật ngữ khó thì BẮT BUỘC phải giải thích bằng ví dụ gần gũi.
    3. Hướng dẫn Bài tập (Dạng bài: {question_type}): Nếu học sinh hỏi bài tập, hãy GỢI Ý TỪNG BƯỚC để em tự suy nghĩ, KHÔNG đưa ngay đáp án hoàn chỉnh.
    4. An toàn là trên hết: KHÔNG hướng dẫn bất kỳ thí nghiệm hay hành vi nguy hiểm nào. Nếu là tình huống mất an toàn (như làm vỡ nhiệt kế), hãy dặn dò cẩn thận và nhắc em báo ngay cho người lớn.
    5. Độ dài: Câu trả lời ngắn gọn, vừa đủ, không dài dòng tràn lan.

    --- DỮ LIỆU SÁCH GIÁO KHOA ---
    {context_text}
    ------------------------------

    Câu hỏi của học sinh: "{user_query}"
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=pedagogical_prompt
        )
        print("\n CHATBOT GIÁO VIÊN PHẢN HỒI:")
        print(response.text)
    except Exception as e:
        print(f" Lỗi sinh câu trả lời: {e}")

if __name__ == "__main__":
    print("=== HOÀN THÀNH PIPELINE RAG (TỪ BƯỚC 1 ĐẾN BƯỚC 7) ===")

    rag_chatbot_pipeline("Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?")
    print("\n" + "="*60)

    rag_chatbot_pipeline("Cần làm gì nếu không may làm vỡ nhiệt kế thủy ngân?")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== HOÀN THÀNH PIPELINE RAG (TỪ BƯỚC 1 ĐẾN BƯỚC 7) ===

 Học sinh hỏi: 'Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?'
 [1/3] Đang phân tích câu hỏi...
   -> Môn: Khoa học tự nhiên | Dạng: Vận dụng
 [2/3] Đang tìm kiếm kiến thức trong kho sách...
   -> Đã tìm thấy 3 đoạn thông tin khớp.
⏳ [3/3] Đang sinh câu trả lời theo chuẩn Sư phạm...

 CHATBOT GIÁO VIÊN PHẢN HỒI:
Chào em! Đây là một tình huống rất quan trọng cần lưu ý nhé. 

Nếu không may làm vỡ nhiệt kế, em nhớ thật kỹ các điều sau đây:
1. **Tuyệt đối không được sờ tay vào thủy ngân** (vì thủy ngân rất độc và dễ bay hơi).
2. Phải **báo ngay cho giáo viên hoặc người lớn** xung quanh để được giúp đỡ và xử lý an toàn nhé.

Em có nhớ rõ hai điều này chưa nào?


 Học sinh hỏi: 'Cần làm gì nếu không may làm vỡ nhiệt kế thủy ngân?'
 [1/3] Đang phân tích câu hỏi...
   -> Môn: Khoa học tự nhiên | Dạng: Vận dụng
 [2/3] Đang tìm kiếm kiến thức trong kho sách...
   -> Đã tìm thấy 3 đoạn thông tin khớp.
⏳ [3/3] Đang sinh câu trả lời theo chuẩ

In [14]:
import os
import json
import re
import chromadb
from google import genai
from sentence_transformers import SentenceTransformer

API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJtGZUsnS7RD1hnST_ZxIA"
client = genai.Client(api_key=API_KEY)

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_collection(name="grade6_knowledge_base")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def analyze_student_query(query: str) -> dict:
    prompt = f"""
    Bạn là module phân tích dữ liệu cho Chatbot giáo dục lớp 6.
    Nhiệm vụ: Phân tích câu hỏi của học sinh: "{query}"

    QUY TẮC PHÂN LOẠI DẠNG CÂU HỎI (question_type):
    - "Khái niệm": Học sinh hỏi "là gì", "tại sao", "như thế nào", "phải làm sao khi..." (Hỏi tình huống thực tế/kiến thức chung).
    - "Bài tập đơn giản": Hỏi cách giải bài tập cơ bản, tính toán đơn giản.
    - "Bài tập khó": Bài tập nâng cao, nhiều bước làm.
    - "Yêu cầu đáp án trực tiếp": Học sinh dùng từ khóa xin xỏ như "cho em xin đáp án", "giải hộ em câu...", "cho em xem lời giải".

    Trả về MỘT CHUỖI JSON DUY NHẤT (không dùng markdown code block):
    {{
        "subject": "Công nghệ" hoặc "Khoa học tự nhiên" (Nếu không rõ ghi "Unknown"),
        "topic": "Tên chủ đề ngắn gọn" (Nếu không rõ ghi "Unknown"),
        "question_type": "Khái niệm" / "Bài tập đơn giản" / "Bài tập khó" / "Yêu cầu đáp án trực tiếp",
        "difficulty": "Dễ" / "Trung bình" / "Khó",
        "intent": "Giải thích" / "Xin đáp án" / "Cần gợi ý" / "Trả lời thử"
    }}
    """
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )
        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        return json.loads(match.group(0)) if match else None
    except Exception:
        return None

def retrieve_knowledge(query: str, top_k: int = 3, distance_threshold: float = 20.0) -> list:
    query_embedding = embedding_model.encode([query]).tolist()
    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        retrieved_chunks = []
        if results and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                dist = results['distances'][0][i]
                if dist <= distance_threshold:
                    retrieved_chunks.append(results['documents'][0][i])
        return retrieved_chunks
    except Exception:
        return []

def rag_chatbot_pipeline(user_query: str):
    print(f"\n Học sinh hỏi: '{user_query}'")
    print(" [1/3] Đang phân tích câu hỏi...")

    analysis = analyze_student_query(user_query)

    if not analysis or analysis.get("subject") == "Unknown":
        print("\n Chatbot: 'Chào em! Câu hỏi của em hơi chung chung một chút. Em đang muốn hỏi kiến thức thuộc môn Công nghệ hay Khoa học tự nhiên lớp 6 vậy nhỉ? Cho thầy/cô biết thêm chi tiết để hỗ trợ em nhé!'")
        return

    q_type = analysis.get("question_type", "Khái niệm")
    print(f"   -> Môn: {analysis['subject']} | Dạng: {q_type} | Ý định: {analysis.get('intent')}")
    print(" [2/3] Đang tìm kiếm kiến thức...")

    context_chunks = retrieve_knowledge(user_query)

    if not context_chunks:
        print("\n Chatbot: 'Chào em! Kiến thức này hiện chưa có trong kho bài học lớp 6 của thầy/cô hoặc vượt quá chương trình học hiện tại. Em có thể hỏi lại thầy cô bộ môn trên lớp để được giải đáp chi tiết hơn nhé!'")
        return

    print(f"   -> Đã tìm thấy {len(context_chunks)} đoạn thông tin khớp.")
    print(" [3/3] Đang định dạng câu trả lời sư phạm...")

    context_text = "\n\n".join([f"- {c}" for c in context_chunks])

    pedagogical_prompt = f"""
    Bạn là một Trợ giảng AI thông minh, dịu dàng dành cho học sinh lớp 6.
    Xưng hô: "thầy/cô" - "em".

    === PHÂN LOẠI TÌNH HUỐNG BÀI TẬP (MỤC 3.10) ===
    Dạng câu hỏi được phân tích: [{q_type}]
    - Nếu dạng là "Yêu cầu đáp án trực tiếp": Lịch sự hỏi xem em đã thử tự làm chưa, sau đó chỉ đưa ra gợi ý/hướng dẫn, KHÔNG cho ngay đáp án hoàn chỉnh.
    - Nếu dạng là "Bài tập đơn giản" hoặc "Bài tập khó": Gợi ý từng bước (Bước 1, Bước 2...) để em tự suy nghĩ và giải bài.
    - Nếu dạng là "Khái niệm": Giải thích trực tiếp và BẮT BUỘC trình bày theo CẤU TRÚC 5 PHẦN dưới đây.

    === CẤU TRÚC CÂU TRẢ LỜI 5 PHẦN (CHỈ ÁP DỤNG KHI GIẢI THÍCH KHÁI NIỆM/TÌNH HUỐNG) ===
    1. Trả lời ngắn gọn: (Bắt đầu bằng "Em có thể hiểu đơn giản là...")
    2. Giải thích dễ hiểu: (Giải thích nội dung bằng câu từ đơn giản)
    3. Ví dụ thực tế: (Bắt đầu bằng "Ví dụ: ...")
    4. Ghi nhớ chính: (Bắt đầu bằng "Ghi nhớ: ...")
    5. Câu hỏi kiểm tra: (Bắt đầu bằng "Em thử trả lời câu hỏi nhỏ này nhé: ...")

    --- DỮ LIỆU SÁCH GIÁO KHOA CHUẨN ---
    {context_text}
    ------------------------------------

    YÊU CẦU AN TOÀN:
    - Chỉ sử dụng kiến thức trong DỮ LIỆU SÁCH GIÁO KHOA trên.
    - Tuyệt đối KHÔNG hướng dẫn bất kỳ thí nghiệm hay hành vi nguy hiểm nào.
    - Nhắc nhở cẩn thận nếu hỏi tình huống vỡ nhiệt kế.

    Câu hỏi của học sinh: "{user_query}"
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=pedagogical_prompt
        )
        print("\n CHATBOT TRỢ GIẢNG PHẢN HỒI:")
        print(response.text)
    except Exception as e:
        print(f" Lỗi sinh câu trả lời: {e}")

if __name__ == "__main__":
    print("=== PIPELINE BƯỚC 8 (ĐÃ CẬP NHẬT HOÀN CHỈNH)")

    rag_chatbot_pipeline("Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?")

    print("\n" + "="*70)

    rag_chatbot_pipeline("Cô cho em xin đáp án câu hỏi ôn tập vỡ nhiệt kế thủy ngân với ạ!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== PIPELINE BƯỚC 8 (ĐÃ CẬP NHẬT HOÀN CHỈNH)

 Học sinh hỏi: 'Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?'
 [1/3] Đang phân tích câu hỏi...
   -> Môn: Khoa học tự nhiên | Dạng: Khái niệm | Ý định: Giải thích
 [2/3] Đang tìm kiếm kiến thức...
   -> Đã tìm thấy 3 đoạn thông tin khớp.
 [3/3] Đang định dạng câu trả lời sư phạm...

 CHATBOT TRỢ GIẢNG PHẢN HỒI:
Chào em, cô rất sẵn sàng giải đáp thắc mắc cho em nhé! Thủy ngân là một chất rất quan trọng nhưng cũng rất nguy hiểm, vì vậy câu hỏi của em rất thực tế và cần thiết đấy. 

Dưới đây là phần giải thích dành cho em:

1. Trả lời ngắn gọn: 
Em có thể hiểu đơn giản là: Nếu không may làm vỡ nhiệt kế thủy ngân, tuyệt đối không được sờ tay vào thủy ngân và phải báo ngay cho giáo viên hoặc người lớn để được xử lý an toàn.

2. Giải thích dễ hiểu: 
Thủy ngân bên trong nhiệt kế y tế là một chất độc và rất dễ bay hơi vào không khí. Nếu mình chạm tay vào hoặc hít phải hơi của nó, cơ thể sẽ bị ảnh hưởng xấu đến sức khỏe. Vì vậy, khi xảy ra sự cố v

In [15]:
import os
import json
import re
import chromadb
from google import genai
from sentence_transformers import SentenceTransformer

# 1. CẤU HÌNH & KHỞI TẠO HỆ THỐNG
API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJt6RZUsnS7RD1hnST_ZxIA"
client = genai.Client(api_key=API_KEY)

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")

# Kết nối Vector DB & Embedding Model
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = chroma_client.get_collection(name="grade6_knowledge_base")
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 2. MODULE AN TOÀN NỘI DUNG (SAFETY GUARDRAIL)
def safety_guardrail(query: str) -> dict:
    """
    Kiểm tra 7 tiêu chí an toàn dành cho học sinh lớp 6.
    """
    safety_prompt = f"""
    Bạn là Module Kiểm duyệt An toàn Nội dung cho Chatbot Giáo dục Lớp 6.
    Kiểm tra câu hỏi của học sinh: "{query}"

    CÁC QUY TẮC AN TOÀN BẮT BUỘC (VIOLATION CODES):
    1. DANGEROUS_EXPERIMENT: Hướng dẫn thí nghiệm nguy hiểm, cháy nổ, hóa chất độc hại tại nhà.
    2. AGE_INAPPROPRIATE: Nội dung không phù hợp lứa tuổi 11-12 (người lớn, nhạy cảm, bạo lực, độc hại).
    3. MEDICAL_ADVICE: Tư vấn y tế cá nhân (chẩn đoán bệnh, kê đơn thuốc, tư vấn điều trị).
    4. CHEATING: Khuyến khích hoặc gian lận học tập (ví dụ: làm hộ bài kiểm tra đang diễn ra).
    5. PII_SOLICITATION: Thu thập, yêu cầu hoặc học sinh tự đưa ra thông tin cá nhân nhạy cảm (địa chỉ nhà, số điện thoại, tên trường cụ thể, mật khẩu...).

    Trả về MỘT CHUỖI JSON DUY NHẤT (không dùng markdown code block):
    {{
        "is_safe": True hoặc False,
        "violation_code": "NONE" hoặc tên mã lỗi ở trên,
        "advice_message": "Lời dặn dò lịch sự, ấm áp từ thầy/cô nếu vi phạm (nếu an toàn ghi '')"
    }}
    """
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=safety_prompt
        )
        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        return {"is_safe": True, "violation_code": "NONE", "advice_message": ""}
    except Exception:

        return {"is_safe": True, "violation_code": "NONE", "advice_message": ""}

def analyze_student_query(query: str) -> dict:
    prompt = f"""
    Bạn là module phân tích dữ liệu cho Chatbot giáo dục lớp 6.
    Nhiệm vụ: Phân tích câu hỏi của học sinh: "{query}"

    QUY TẮC PHÂN LOẠI DẠNG CÂU HỎI (question_type):
    - "Khái niệm": Học sinh hỏi "là gì", "tại sao", "như thế nào", "phải làm sao khi..." (Hỏi tình huống thực tế/kiến thức chung).
    - "Bài tập đơn giản": Hỏi cách giải bài tập cơ bản, tính toán đơn giản.
    - "Bài tập khó": Bài tập nâng cao, nhiều bước làm.
    - "Yêu cầu đáp án trực tiếp": Học sinh dùng từ khóa xin xỏ như "cho em xin đáp án", "giải hộ em câu...", "cho em xem lời giải".

    Trả về MỘT CHUỖI JSON DUY NHẤT (không dùng markdown code block):
    {{
        "subject": "Công nghệ" hoặc "Khoa học tự nhiên" (Nếu không rõ ghi "Unknown"),
        "topic": "Tên chủ đề ngắn gọn" (Nếu không rõ ghi "Unknown"),
        "question_type": "Khái niệm" / "Bài tập đơn giản" / "Bài tập khó" / "Yêu cầu đáp án trực tiếp",
        "difficulty": "Dễ" / "Trung bình" / "Khó",
        "intent": "Giải thích" / "Xin đáp án" / "Cần gợi ý" / "Trả lời thử"
    }}
    """
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )
        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        return json.loads(match.group(0)) if match else None
    except Exception:
        return None

def retrieve_knowledge(query: str, top_k: int = 3, distance_threshold: float = 20.0) -> list:
    query_embedding = embedding_model.encode([query]).tolist()
    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        retrieved_chunks = []
        if results and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                dist = results['distances'][0][i]
                if dist <= distance_threshold:
                    retrieved_chunks.append(results['documents'][0][i])
        return retrieved_chunks
    except Exception:
        return []

def rag_chatbot_pipeline(user_query: str):
    print(f"\n👤 Học sinh hỏi: '{user_query}'")

    print(" [0/3] Đang kiểm tra An toàn nội dung...")
    safety_check = safety_guardrail(user_query)

    if not safety_check.get("is_safe", True):
        v_code = safety_check.get("violation_code")
        print(f" PHÁT HIỆN VI PHẠM AN TOÀN: [{v_code}]")
        print("\n CHATBOT TRỢ GIẢNG PHẢN HỒI:")

        if v_code == "MEDICAL_ADVICE":
            print("Chào em! Thầy/cô không phải là bác sĩ nên không thể tư vấn y tế hay kê đơn thuốc cho em được. Nếu em cảm thấy không khỏe, hãy báo ngay cho cha mẹ hoặc thầy cô y tế ở trường để được chăm sóc tốt nhất nhé!")
        elif v_code == "DANGEROUS_EXPERIMENT":
            print("Chào em! Để đảm bảo an toàn tuyệt đối, thầy/cô không thể hướng dẫn các thí nghiệm nguy hiểm hoặc dùng hóa chất lạ tại nhà. Em chỉ nên thực hiện các thí nghiệm có trong sách giáo khoa dưới sự giám sát của giáo viên nhé!")
        elif v_code == "PII_SOLICITATION":
            print("Chào em! Để bảo vệ an toàn trên mạng, em tuyệt đối không nên chia sẻ các thông tin cá nhân như địa chỉ nhà, số điện thoại hay tên trường học nhé. Thầy/cô cũng sẽ không bao giờ hỏi các thông tin này của em đâu!")
        else:
            print(f"Chào em! {safety_check.get('advice_message', 'Nội dung này chưa phù hợp để thảo luận. Em hãy hỏi những câu hỏi liên quan đến bài học lớp 6 nhé!')}")
        return

    print(" Kiểm tra an toàn: ĐẠT")
    print(" [1/3] Đang phân tích câu hỏi...")
    analysis = analyze_student_query(user_query)

    if not analysis or analysis.get("subject") == "Unknown":
        print("\n Chatbot: 'Chào em! Câu hỏi của em hơi chung chung một chút. Em đang muốn hỏi kiến thức thuộc môn Công nghệ hay Khoa học tự nhiên lớp 6 vậy nhỉ? Cho thầy/cô biết thêm chi tiết để hỗ trợ em nhé!'")
        return

    q_type = analysis.get("question_type", "Khái niệm")
    print(f"   -> Môn: {analysis['subject']} | Dạng: {q_type} | Ý định: {analysis.get('intent')}")
    print(" [2/3] Đang tìm kiếm kiến thức...")

    context_chunks = retrieve_knowledge(user_query)

    if not context_chunks:
        print("\n Chatbot: 'Chào em! Kiến thức này hiện chưa có trong kho bài học lớp 6 của thầy/cô hoặc vượt quá chương trình học hiện tại. Em có thể hỏi lại thầy cô bộ môn trên lớp để được giải đáp chi tiết hơn nhé!'")
        return

    print(f"   -> Đã tìm thấy {len(context_chunks)} đoạn thông tin khớp.")
    print(" [3/3] Đang định dạng câu trả lời sư phạm...")

    context_text = "\n\n".join([f"- {c}" for c in context_chunks])

    pedagogical_prompt = f"""
    Bạn là một Trợ giảng AI thông minh, dịu dàng dành cho học sinh lớp 6.
    Xưng hô: "thầy/cô" - "em".

    === PHÂN LOẠI TÌNH HUỐNG BÀI TẬP (MỤC 3.10) ===
    Dạng câu hỏi được phân tích: [{q_type}]
    - Nếu dạng là "Yêu cầu đáp án trực tiếp": Lịch sự hỏi xem em đã thử tự làm chưa, sau đó chỉ đưa ra gợi ý/hướng dẫn, KHÔNG cho ngay đáp án hoàn chỉnh.
    - Nếu dạng là "Bài tập đơn giản" hoặc "Bài tập khó": Gợi ý từng bước (Bước 1, Bước 2...) để em tự suy nghĩ và giải bài.
    - Nếu dạng là "Khái niệm": Giải thích trực tiếp và BẮT BUỘC trình bày theo CẤU TRÚC 5 PHẦN dưới đây.

    === CẤU TRÚC CÂU TRẢ LỜI 5 PHẦN (CHỈ ÁP DỤNG KHI GIẢI THÍCH KHÁI NIỆM/TÌNH HUỐNG) ===
    1. Trả lời ngắn gọn: (Bắt đầu bằng "Em có thể hiểu đơn giản là...")
    2. Giải thích dễ hiểu: (Giải thích nội dung bằng câu từ đơn giản)
    3. Ví dụ thực tế: (Bắt đầu bằng "Ví dụ: ...")
    4. Ghi nhớ chính: (Bắt đầu bằng "Ghi nhớ: ...")
    5. Câu hỏi kiểm tra: (Bắt đầu bằng "Em thử trả lời câu hỏi nhỏ này nhé: ...")

    --- DỮ LIỆU SÁCH GIÁO KHOA CHUẨN ---
    {context_text}
    ------------------------------------

    YÊU CẦU AN TOÀN NGHIÊM NGẶT:
    - Tuyệt đối KHÔNG yêu cầu hoặc gợi ý học sinh cung cấp địa chỉ, số điện thoại hay thông tin cá nhân.
    - Không tư vấn y tế cá nhân.
    - Không hướng dẫn thí nghiệm nguy hiểm.

    Câu hỏi của học sinh: "{user_query}"
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=pedagogical_prompt
        )
        print("\n CHATBOT TRỢ GIẢNG PHẢN HỒI:")
        print(response.text)
    except Exception as e:
        print(f" Lỗi sinh câu trả lời: {e}")

if __name__ == "__main__":
    print("=== TEST MODULE AN TOÀN NỘI DUNG & PIPELINE ===")

    rag_chatbot_pipeline("Em bị đau bụng và sốt thì nên uống thuốc gì hả cô?")
    print("\n" + "="*70)

    rag_chatbot_pipeline("Cô cho em xin số điện thoại và địa chỉ nhà để em đến hỏi bài nhé!")
    print("\n" + "="*70)

    rag_chatbot_pipeline("Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== TEST MODULE AN TOÀN NỘI DUNG & PIPELINE ===

👤 Học sinh hỏi: 'Em bị đau bụng và sốt thì nên uống thuốc gì hả cô?'
 [0/3] Đang kiểm tra An toàn nội dung...
 PHÁT HIỆN VI PHẠM AN TOÀN: [MEDICAL_ADVICE]

 CHATBOT TRỢ GIẢNG PHẢN HỒI:
Chào em! Thầy/cô không phải là bác sĩ nên không thể tư vấn y tế hay kê đơn thuốc cho em được. Nếu em cảm thấy không khỏe, hãy báo ngay cho cha mẹ hoặc thầy cô y tế ở trường để được chăm sóc tốt nhất nhé!


👤 Học sinh hỏi: 'Cô cho em xin số điện thoại và địa chỉ nhà để em đến hỏi bài nhé!'
 [0/3] Đang kiểm tra An toàn nội dung...
 PHÁT HIỆN VI PHẠM AN TOÀN: [PII_SOLICITATION]

 CHATBOT TRỢ GIẢNG PHẢN HỒI:
Chào em! Để bảo vệ an toàn trên mạng, em tuyệt đối không nên chia sẻ các thông tin cá nhân như địa chỉ nhà, số điện thoại hay tên trường học nhé. Thầy/cô cũng sẽ không bao giờ hỏi các thông tin này của em đâu!


👤 Học sinh hỏi: 'Nếu lỡ làm vỡ nhiệt kế thì phải làm sao hả cô?'
 [0/3] Đang kiểm tra An toàn nội dung...
 Kiểm tra an toàn: ĐẠT
 [1/3] Đang phân 

In [16]:
!pip install -q chromadb google-genai sentence-transformers gradio

In [17]:
pip install -U google-generativeai

In [18]:
pip install -U google-genai

In [19]:
import os
import chromadb
import gradio as gr
from google import genai
from sentence_transformers import SentenceTransformer

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

API_KEY = "AQ.Ab8RN6JGPkZ3w6q_4uXK_zfR58kDJt6RZUsnS7RD1hnST_ZxIA"

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"

DRIVE_FOLDER = "/content/drive/MyDrive/ĐỒ ÁN CHUYÊN NGÀNH"
CHROMA_DB_PATH = os.path.join(DRIVE_FOLDER, "chroma_db")

try:
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
    collection = chroma_client.get_collection(name="grade6_knowledge_base")
except Exception:
    collection = None

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def retrieve_knowledge(query: str, subject_filter: str, top_k: int = 3, distance_threshold: float = 20.0) -> list:
    if not collection:
        return []

    query_embedding = embedding_model.encode([query]).tolist()
    where_clause = {"subject": subject_filter} if subject_filter in ["Khoa học tự nhiên", "Công nghệ"] else None

    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k,
            where=where_clause
        )
        retrieved_chunks = []
        if results and results.get('documents') and results['documents'][0]:
            for i in range(len(results['documents'][0])):
                dist = results['distances'][0][i]
                if dist <= distance_threshold:
                    retrieved_chunks.append(results['documents'][0][i])

        if not retrieved_chunks and where_clause:
            results = collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )
            if results and results.get('documents') and results['documents'][0]:
                for i in range(len(results['documents'][0])):
                    dist = results['distances'][0][i]
                    if dist <= distance_threshold:
                        retrieved_chunks.append(results['documents'][0][i])

        return retrieved_chunks
    except Exception:
        return []

def process_chat(user_query: str, selected_subject: str, chat_history: list):
    chat_history = chat_history or []
    if not user_query or not user_query.strip():
        return chat_history, "", ""

    context_chunks = retrieve_knowledge(user_query, selected_subject)

    if not context_chunks:
        context_text = "Hãy sử dụng kiến thức chuẩn xác theo chương trình Sách giáo khoa Lớp 6 của Bộ GD&ĐT Việt Nam để trả lời câu hỏi này một cách dễ hiểu nhất."
    else:
        context_text = "\n\n".join([f"- {c}" for c in context_chunks])

    master_prompt = f"""
    Bạn là Trợ giảng AI dịu dàng, thông minh dạy môn {selected_subject} Lớp 6. Xưng hô: "thầy/cô" - "em".

    === BƯỚC 1: KIỂM TRUYỆT AN TOÀN ===
    Nếu câu hỏi của học sinh thuộc các vi phạm sau:
    - Thí nghiệm nguy hiểm, cháy nổ, hóa chất độc hại.
    - Nội dung không phù hợp lứa tuổi 11-12.
    - Tư vấn y tế cá nhân (chẩn đoán bệnh, kê đơn thuốc).
    - Thu thập thông tin cá nhân (địa chỉ, SĐT).
    -> Hãy từ chối một cách khéo léo, dặn dò lịch sự và KHÔNG trả lời kiến thức.

    === BƯỚC 2: TRẢ LỜI SƯ PHẠM (Nếu an toàn) ===
    - Nếu là câu hỏi "Khái niệm": Trình bày theo CẤU TRÚC 5 PHẦN:
      1. Trả lời ngắn gọn: (Bắt đầu: "Em có thể hiểu đơn giản là...")
      2. Giải thích dễ hiểu: (...)
      3. Ví dụ thực tế: (Bắt đầu: "Ví dụ: ...")
      4. Ghi nhớ chính: (Bắt đầu: "Ghi nhớ: ...")
      5. Câu hỏi kiểm tra: (Bắt đầu: "Em thử trả lời câu hỏi nhỏ này nhé: ...")
    - Nếu là "Bài tập": Gợi ý từng bước, KHÔNG làm hộ toàn bộ đáp án trực tiếp.

    --- DỮ LIỆU SÁCH GIÁO KHOA ---
    {context_text}
    ------------------------------
    Câu hỏi học sinh: "{user_query}"
    """

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=master_prompt,
            config={"temperature": 0.2}
        )
        reply = response.text
    except Exception as e:
        reply = f"Có lỗi nhỏ khi kết nối với AI: {e}"

    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": reply})
    return chat_history, reply, context_text

def handle_action_button(action_type: str, last_reply: str, last_context: str, chat_history: list):
    chat_history = chat_history or []

    action_labels = {
        "easier": "Giải thích dễ hơn giúp em với!",
        "example": "Cho em xin thêm ví dụ thực tế ạ!",
        "quiz": "Tạo câu hỏi trắc nghiệm luyện tập cho em!",
        "not_understand": "Em vẫn chưa hiểu rõ đoạn này..."
    }
    user_label = action_labels.get(action_type, "Hỗ trợ thêm")

    if not last_reply:
        bot_reply = "Em hãy đặt câu hỏi học tập ở ô trên trước khi dùng các nút hỗ trợ này nhé!"
        chat_history.append({"role": "user", "content": user_label})
        chat_history.append({"role": "assistant", "content": bot_reply})
        return chat_history, ""

    prompts = {
        "easier": f"Dựa vào nội dung vừa giải thích:\n'{last_reply}'\nHãy giải thích lại ngắn hơn nữa, dùng hình ảnh so sánh cực kỳ quen thuộc để học sinh 11 tuổi dễ hình dung nhất.",
        "example": f"Dựa vào kiến thức này:\n'{last_reply}'\nHãy cho thêm 2 ví dụ thực tế cực kỳ sinh động và gần gũi trong đời sống hằng ngày của học sinh.",
        "quiz": f"Dựa vào kiến thức bài học:\n'{last_reply}'\nHãy tạo 1 câu hỏi trắc nghiệm ôn tập nhỏ (có 4 đáp án A, B, C, D) để em tự kiểm tra xem mình đã hiểu bài chưa. Chưa cho biết đáp án ngay.",
        "not_understand": f"Học sinh báo 'Em chưa hiểu' nội dung này:\n'{last_reply}'\nHãy kiên nhẫn chia nhỏ nội dung ra, giải thích lại từng bước một thật chậm rãi, ấm áp và động viên em."
    }

    selected_prompt = prompts.get(action_type, "")
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=selected_prompt,
            config={"temperature": 0.1}
        )
        bot_reply = response.text
    except Exception as e:
        bot_reply = f"Không thể khởi tạo phản hồi: {e}"

    chat_history.append({"role": "user", "content": user_label})
    chat_history.append({"role": "assistant", "content": bot_reply})
    return chat_history, bot_reply

def submit_feedback(feedback_type: str):
    if feedback_type == "helpful":
        return "Cảm ơn em nhiều nha! Thầy/cô rất vui vì bài giải này giúp ích cho em!"
    else:
        return "Cảm ơn em đã góp ý! Thầy/cô sẽ cố gắng giải thích rõ ràng và hay hơn ở lần sau nhé!"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [20]:
import gradio as gr

custom_css = """
/* Nền tổng thể Pastel sáng */
body, .gradio-container {
    background: linear-gradient(135deg, #e0f2fe 0%, #fef3c7 50%, #fce7f3 100%) !important;
    font-family: 'Segoe UI', Roboto, sans-serif !important;
}

/* Header chính */
.header-box {
    background: white;
    padding: 16px;
    border-radius: 20px;
    text-align: center;
    box-shadow: 0 4px 15px rgba(2, 132, 199, 0.08);
    border: 3px solid #7dd3fc;
    margin-bottom: 16px;
}
.header-box h1 {
    color: #0284c7 !important;
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    margin-bottom: 4px !important;
}
.header-box p {
    color: #0369a1 !important;
    font-size: 1.05rem !important;
    font-weight: 600 !important;
}

/* Ép các Card/Block về nền trắng viền xanh dịu mát */
.gr-block, .gr-panel, .gr-box {
    background-color: #ffffff !important;
    border-radius: 16px !important;
    border: 2px solid #bae6fd !important;
    box-shadow: 0 4px 12px rgba(0,0,0,0.03) !important;
}

/* Ép màu chữ tiêu đề & label rõ nét, không bị chìm */
label, span, h3, h4, .gr-form label {
    color: #0c4a6e !important;
    font-weight: 700 !important;
}

/*  ĐỔI MÀU & ĐỊNH DẠNG NÚT GỢI Ý CÂU HỎI (gr.Examples) */
.dataset-button, button.gallery-item, div[data-testid="dataset-button"] {
    background: #f0f9ff !important;
    border: 2px solid #0284c7 !important;
    color: #0369a1 !important;
    font-size: 0.95rem !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
    padding: 8px 12px !important;
    margin: 4px 2px !important;
    transition: all 0.2s ease-in-out !important;
}
.dataset-button:hover, button.gallery-item:hover {
    background: #0284c7 !important;
    color: #ffffff !important;
    transform: translateY(-2px);
    box-shadow: 0 4px 10px rgba(2, 132, 199, 0.2) !important;
}

/* Khung Chatbot */
.gr-chatbot {
    background-color: #ffffff !important;
    border-radius: 16px !important;
}

/* Các nút chức năng nhanh bên dưới */
.btn-easier {
    background: linear-gradient(135deg, #4ade80, #22c55e) !important;
    color: white !important; font-weight: bold !important;
    border-radius: 12px !important; border: none !important;
}
.btn-example {
    background: linear-gradient(135deg, #fbbf24, #f59e0b) !important;
    color: white !important; font-weight: bold !important;
    border-radius: 12px !important; border: none !important;
}
.btn-quiz {
    background: linear-gradient(135deg, #c084fc, #a855f7) !important;
    color: white !important; font-weight: bold !important;
    border-radius: 12px !important; border: none !important;
}
.btn-not-understand {
    background: linear-gradient(135deg, #f87171, #ef4444) !important;
    color: white !important; font-weight: bold !important;
    border-radius: 12px !important; border: none !important;
}

button:hover {
    transform: translateY(-2px);
    transition: all 0.2s ease-in-out;
}
"""

theme = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="amber",
    neutral_hue="slate"
)

with gr.Blocks() as demo:
    current_reply_state = gr.State("")
    current_context_state = gr.State("")

    gr.HTML(
        """
        <div class="header-box">
            <h1>LEO THÔNG THÁI</h1>
            <p>Đồng hành cùng em khám phá Khoa học tự nhiên & Công nghệ 6 </p>
        </div>
        """
    )

    # Căn chỉnh chiều cao 2 cột cân bằng
    with gr.Row(equal_height=True):
        with gr.Column(scale=4):
            subject_dropdown = gr.Dropdown(
                choices=["Khoa học tự nhiên", "Công nghệ"],
                value="Khoa học tự nhiên",
                label=" Chọn môn học em muốn hỏi",
                interactive=True
            )
            query_input = gr.Textbox(
                placeholder="Nhập câu hỏi và nhấn Enter hoặc nút Gửi...",
                label=" Nhập câu hỏi của em vào đây",
                lines=3
            )
            submit_btn = gr.Button(" Gửi câu hỏi cho Thầy/Cô", variant="primary", size="lg")

            gr.Examples(
                examples=[
                    ["Tế bào là gì hả cô?"],
                    ["Nếu lỡ làm vỡ nhiệt kế thủy ngân thì phải làm sao?"],
                    ["Tại sao phải rửa rau trước khi ăn?"]
                ],
                inputs=query_input,
                label=" Câu hỏi gợi ý cho em"
            )

        with gr.Column(scale=6):
            chatbot_display = gr.Chatbot(
                label=" Góc thảo luận bài học",
                height=460,
                avatar_images=(
                    "https://cdn-icons-png.flaticon.com/512/3135/3135715.png",
                    "https://cdn-icons-png.flaticon.com/512/4712/4712109.png"
                )
            )

    gr.Markdown("###  Em muốn thầy/cô hỗ trợ thêm gì không?")
    with gr.Row():
        btn_easier = gr.Button(" Giải thích dễ hơn", elem_classes=["btn-easier"])
        btn_example = gr.Button(" Cho ví dụ thực tế", elem_classes=["btn-example"])
        btn_quiz = gr.Button(" Tạo câu hỏi trắc nghiệm", elem_classes=["btn-quiz"])
        btn_not_understand = gr.Button(" Em chưa hiểu", elem_classes=["btn-not-understand"])

    gr.Markdown("---")
    with gr.Row():
        gr.Markdown("** Đánh giá bài giải của thầy/cô:**")
        btn_like = gr.Button("Bài giải rất hữu ích", size="sm")
        btn_dislike = gr.Button("Cần giải thích hay hơn", size="sm")

    feedback_output = gr.Markdown("")

    submit_btn.click(
        fn=process_chat,
        inputs=[query_input, subject_dropdown, chatbot_display],
        outputs=[chatbot_display, current_reply_state, current_context_state]
    ).then(fn=lambda: "", outputs=query_input)

    query_input.submit(
        fn=process_chat,
        inputs=[query_input, subject_dropdown, chatbot_display],
        outputs=[chatbot_display, current_reply_state, current_context_state]
    ).then(fn=lambda: "", outputs=query_input)

    btn_easier.click(
        fn=lambda r, c, h: handle_action_button("easier", r, c, h),
        inputs=[current_reply_state, current_context_state, chatbot_display],
        outputs=[chatbot_display, current_reply_state]
    )
    btn_example.click(
        fn=lambda r, c, h: handle_action_button("example", r, c, h),
        inputs=[current_reply_state, current_context_state, chatbot_display],
        outputs=[chatbot_display, current_reply_state]
    )
    btn_quiz.click(
        fn=lambda r, c, h: handle_action_button("quiz", r, c, h),
        inputs=[current_reply_state, current_context_state, chatbot_display],
        outputs=[chatbot_display, current_reply_state]
    )
    btn_not_understand.click(
        fn=lambda r, c, h: handle_action_button("not_understand", r, c, h),
        inputs=[current_reply_state, current_context_state, chatbot_display],
        outputs=[chatbot_display, current_reply_state]
    )

    btn_like.click(fn=lambda: submit_feedback("helpful"), outputs=feedback_output)
    btn_dislike.click(fn=lambda: submit_feedback("not_helpful"), outputs=feedback_output)

if __name__ == "__main__":
    demo.launch(share=True, theme=theme, css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0300bdc420d229af41.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
